# Task 1 — Dataset Investigation: Neyshekar Persian ASR
**Goal:** Complete end-to-end data quality investigation, text normalization, multi-stage deduplication, audio health check, speech rate (WPS/CPS) audit, parallel signal processing (silence/clipping), and distribution analysis on Neyshekar Persian ASR dataset.

## 1. Initial Assumptions Regarding Data Quality
1. **Hallucination-Prone Samples:** Autoregressive models (Whisper) hallucinate on long silences or empty audio.
2. **Duration Outliers:** Audio < 1s lacks context; audio > 30s will be truncated.
3. **Textual Anomalies:** Arabic encodings (`ي`, `ك`) and unlexicalized digits distort tokenization.
4. **Audio-Text Misalignment:** Outliers in Speech Rate metrics (CPS/WPS).
5. **Data Duplication:** Duplicate sentences cause memorization and overfitting.
6. **Sampling Rate Consistency:** Whisper requires 16 kHz mono WAV audio.

In [ ]:
# Supplied at run time rather than written into the notebook, so this runs anywhere.
# Pass --dataset_path on the command line, or set NEYSHEKAR_RAW_DIR before launching
# Jupyter. See src/paths.py for the resolution order.
import sys, os
sys.path.append(os.path.abspath(os.path.join('..', 'src')))
from paths import resolve_paths

DATASET_PATH, OUTPUT_DIR = resolve_paths()
print('dataset:', DATASET_PATH)
print('output :', OUTPUT_DIR)

## 2. Dataset Loading & Text Quality Audit (Raw Transcripts)

In [ ]:
parquet_files = sorted(glob.glob(os.path.join(DATASET_PATH, 'train-*.parquet')))
dfs = [pd.read_parquet(f, columns=['id', 'audio', 'text', 'duration']) for f in parquet_files]
df = pd.concat(dfs, ignore_index=True)
initial_total = len(df)
df['text_raw'] = df['text'].fillna('').astype(str)

df['has_arabic'] = df['text_raw'].apply(has_arabic_chars)
df['has_digits'] = df['text_raw'].apply(has_digits)
raw_arabic_count = df['has_arabic'].sum()
raw_digits_count = df['has_digits'].sum()
raw_duplicates_count = df['text_raw'].duplicated().sum()

print('=== RAW DATASET TEXT AUDIT ===')
print(f'Total Audio Samples:                    {initial_total:,}')
print(f'Transcripts with Arabic Encodings (ي,ك):  {raw_arabic_count:,} ({raw_arabic_count/initial_total*100:.2f}%)')
print(f'Transcripts with Numeric Digits (0-9):   {raw_digits_count:,} ({raw_digits_count/initial_total*100:.2f}%)')
print(f'Raw Exact Duplicate Transcripts:        {raw_duplicates_count:,} ({raw_duplicates_count/initial_total*100:.2f}%)')

## 3. Step 1: Text Normalization (Arabic -> Persian & Digit Lexicalization)

In [ ]:
df['text_step1'] = df['text_raw'].apply(apply_step1_normalization)
remaining_arabic = df['text_step1'].apply(has_arabic_chars).sum()
remaining_digits = df['text_step1'].apply(has_digits).sum()

print('=== POST NORMALIZATION VERIFICATION ===')
print(f'Remaining Arabic Characters (Target: 0): {remaining_arabic:,}')
print(f'Remaining Numeric Digits (Target: 0):    {remaining_digits:,}')

## 4. Are there duplicated transcripts? (3-Stage Deduplication & Acoustic Diversity)

In [ ]:
def compute_audio_hash(audio_dict):
    if isinstance(audio_dict, dict):
        audio_bytes = audio_dict.get('bytes')
        if audio_bytes is not None:
            return hashlib.md5(audio_bytes).hexdigest()
        path = audio_dict.get('path')
        if path:
            return str(path)
    return str(audio_dict)

df['audio_hash'] = df['audio'].apply(compute_audio_hash)
exact_row_dups_mask = df.duplicated(subset=['audio_hash', 'text_raw'], keep='first')
exact_row_dups_dropped = exact_row_dups_mask.sum()
df_stage1 = df[~exact_row_dups_mask].copy().reset_index(drop=True)

df_stage1['text_fingerprint'] = df_stage1['text_step1'].apply(lambda x: remove_spaces_and_zwnj(remove_diacritics(x)))

MAX_COPIES_PER_LONG_TEXT = 3
short_kept_count = 0
long_small_kept_count = 0
long_heavy_kept_count = 0
long_dropped_count = 0
rows_to_drop = []

grouped = df_stage1.groupby('text_fingerprint')
for fp, group_indices in grouped.groups.items():
    count = len(group_indices)
    if count > 1:
        first_text = df_stage1.loc[group_indices[0], 'text_step1']
        is_short = (len(first_text) < 15) or (len(first_text.split()) < 4)
        if is_short:
            short_kept_count += count
        else:
            if count <= MAX_COPIES_PER_LONG_TEXT:
                long_small_kept_count += count
            else:
                drop_indices = group_indices[MAX_COPIES_PER_LONG_TEXT:]
                rows_to_drop.extend(drop_indices)
                long_heavy_kept_count += MAX_COPIES_PER_LONG_TEXT
                long_dropped_count += len(drop_indices)

df_dedup = df_stage1.drop(index=rows_to_drop).copy().reset_index(drop=True)
print(f'Exact Row Duplicates Dropped:       {exact_row_dups_dropped:,}')
print(f'Long Sentence Excess Copies Dropped:{long_dropped_count:,}')
print(f'Remaining after Deduplication:       {len(df_dedup):,} rows')

## 5. Are there invalid audio files? (Audio Health & Duration Outliers)

In [ ]:
corrupted_indices = []
audio_durations = []
for idx, row in df_dedup.iterrows():
    try:
        info = sf.info(io.BytesIO(row['audio']['bytes']))
        audio_durations.append(info.duration)
    except Exception:
        corrupted_indices.append(idx)
        audio_durations.append(np.nan)

df_dedup['audio_duration'] = audio_durations
valid_audio_mask = (df_dedup['audio_duration'] >= 1.0) & (df_dedup['audio_duration'] <= 30.0)
df_clean = df_dedup[valid_audio_mask].copy().reset_index(drop=True)

print(f'Corrupted Audio Files:      {len(corrupted_indices):,}')
print(f'Audio Files Under 1.0s:     {(df_dedup["audio_duration"] < 1.0).sum():,}')
print(f'Audio Files Over 30.0s:    {(df_dedup["audio_duration"] > 30.0).sum():,}')
print(f'Valid Clean Dataset Records:{len(df_clean):,}')

## 6. Distribution Analysis: Audio Duration Histogram

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 6))
ax = sns.histplot(df_clean['audio_duration'], bins=60, kde=True, color='#2b5c8f', edgecolor='black', alpha=0.75)
plt.axvline(x=1.0, color='#e74c3c', linestyle='--', linewidth=2.5, label='Whisper Valid Range (1s - 30s)')
plt.axvline(x=30.0, color='#e74c3c', linestyle='--', linewidth=2.5)
mean_dur = df_clean['audio_duration'].mean()
plt.axvline(x=mean_dur, color='#27ae60', linestyle='-', linewidth=1.8, label=f'Mean Duration ({mean_dur:.2f}s)')
plt.title('Neyshekar Dataset — Audio Duration Distribution', fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Audio Duration (Seconds)', fontsize=12)
plt.ylabel('Number of Audio Samples', fontsize=12)
plt.xlim(0, 32)
plt.legend(loc='upper right', fontsize=11, frameon=True, facecolor='white')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'audio_duration_histogram.png'), dpi=300)
plt.show()

## 7. Speech Rate (WPS / CPS) & Transcript Length Distribution

In [ ]:
df_clean['word_count'] = df_clean['text_step1'].str.split().str.len()
df_clean['char_count'] = df_clean['text_step1'].str.len()
df_clean['wps'] = df_clean['word_count'] / df_clean['audio_duration']
df_clean['cps'] = df_clean['char_count'] / df_clean['audio_duration']

high_wps_count = (df_clean['wps'] > 5.0).sum()
low_cps_count = (df_clean['cps'] < 8.0).sum()

print(f'High WPS Outliers (WPS > 5.0 words/s): {high_wps_count:,}')
print(f'Low CPS Outliers (CPS < 8.0 chars/s):  {low_cps_count:,} ({low_cps_count/len(df_clean)*100:.2f}%)')

plt.figure(figsize=(12, 6))
sns.histplot(df_clean['word_count'], bins=50, kde=True, color='#e67e22', edgecolor='black', alpha=0.75)
plt.axvline(x=df_clean['word_count'].mean(), color='#c0392b', linestyle='--', label=f"Mean ({df_clean['word_count'].mean():.2f} words)")
plt.title('Neyshekar Dataset — Transcript Word Count Distribution', fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Transcript Word Count', fontsize=12)
plt.ylabel('Number of Samples', fontsize=12)
plt.legend(loc='upper right', fontsize=11, frameon=True, facecolor='white')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'transcript_length_histogram.png'), dpi=300)
plt.show()

## 8. Parallel Signal Quality Audit (Absolute Silence & Amplitude Clipping)

In [ ]:
def analyze_signal_batch(audio_bytes_list):
    results = []
    for b in audio_bytes_list:
        try:
            data, sr = sf.read(io.BytesIO(b), dtype='float32')
            if len(data) == 0:
                results.append((True, False))
                continue
            max_amp = float(np.max(np.abs(data)))
            rms = float(np.sqrt(np.mean(data**2)))
            is_silent = (max_amp < 0.001) or (rms < 0.0001)
            is_clipped = (max_amp >= 0.99)
            results.append((is_silent, is_clipped))
        except Exception:
            results.append((True, False))
    return results

audio_bytes_list = [row['audio']['bytes'] for idx, row in df_clean.iterrows()]
num_cores = os.cpu_count() or 4
chunk_size = int(np.ceil(len(audio_bytes_list) / num_cores))
chunks = [audio_bytes_list[i:i+chunk_size] for i in range(0, len(audio_bytes_list), chunk_size)]

parallel_results = Parallel(n_jobs=num_cores)(delayed(analyze_signal_batch)(chunk) for chunk in chunks)
flattened = [item for sublist in parallel_results for item in sublist]

silent_count = sum(r[0] for r in flattened)
clipped_count = sum(r[1] for r in flattened)

print('=== SIGNAL INTEGRITY RESULTS ===')
print(f'Absolute Silence Audio Files: {silent_count:,}')
print(f'Clipped / Distorted Files:    {clipped_count:,} ({clipped_count/len(df_clean)*100:.2f}%)')

## 9. Critical ML Question: Causes of Potential Training Instability
**Question:** *If training becomes unstable, which characteristics of this dataset would you investigate first?*

**Answer:**
1. **Amplitude Clipping & High Gain (22.10% of samples):** Peak volume clipping at 0 dB causes extreme gradient spikes in Whisper's convolutional encoder.
2. **Low CPS / Long Silences (27.92% of samples):** Long silent padding or slow speech triggers autoregressive decoder looping and speech hallucinations.
3. **Conversational / Informal Speech (34.86% informal):** Colloquial spoken forms conflicting with standard written Persian tokens.